# Advanced Task-Vector Merging Experiments (Qwen3-1.7B)

This notebook extends `merging.ipynb` by keeping the same task-vector setup (base/IF/Math checkpoints) and adding three nonlinear merge families:

1. Coordinate-wise gating
2. Norm-aware nonlinear blend
3. Subspace-aware merge

Core setup:
- Base: `Qwen/Qwen3-1.7B`
- IF tuned: `/mnt/ddn/vuvlm/geeho/nemotron_cascade_output/Qwen3-1.7B-ifrl_ifeval/global_step_50/actor/huggingface`
- Math tuned: `/mnt/ddn/vuvlm/geeho/nemotron_cascade_output/Qwen3-1.7B-math/stage2/global_step_40/actor/huggingface`


## Method Equations

### 1) Coordinate-wise gating
For each parameter coordinate `i`:

- `w_i = sigmoid(a * |Δ_if,i| - b * |Δ_math,i| + c * agree_i)`
- `θ_i = θ_0,i + w_i * Δ_if,i + (1 - w_i) * Δ_math,i`

where `agree_i` is a sign-agreement signal (`+1`, `0`, `-1`).

### 2) Norm-aware nonlinear blend
- Saturate large deltas with `tanh` or `clip` to suppress over-dominance.
- Blend saturated task vectors using inverse-norm weights.

### 3) Subspace-aware merge
- Build a low-dimensional basis from IF/Math deltas per tensor (common + orthogonal/conflict axis).
- Combine common and conflict coefficients with different strengths.


In [ ]:
from __future__ import annotations

import gc
import json
from dataclasses import asdict, dataclass
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Mapping, Optional, Tuple

import torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

# --------------------------------------------------------------------------------------
# Global experiment configuration
# --------------------------------------------------------------------------------------
# Keep the exact checkpoint trio requested by the experiment.
BASE_MODEL_ID = "Qwen/Qwen3-1.7B"
IF_MODEL_PATH = Path("/mnt/ddn/vuvlm/geeho/nemotron_cascade_output/Qwen3-1.7B-ifrl_ifeval/global_step_50/actor/huggingface")
MATH_MODEL_PATH = Path("/mnt/ddn/vuvlm/geeho/nemotron_cascade_output/Qwen3-1.7B-math/stage2/global_step_40/actor/huggingface")

# Store all merged checkpoints and metadata under one dedicated root.
OUTPUT_ROOT = Path("/mnt/ddn/vuvlm/geeho/nemotron_cascade_output/Qwen3-1.7B-advanced-merge")
MODEL_DTYPE = torch.bfloat16

# Set a deterministic seed for reproducibility of any stochastic operation.
# Current merge implementations are deterministic, but we set this explicitly
# to make the notebook robust to future random components.
torch.manual_seed(42)

for required_path in [IF_MODEL_PATH, MATH_MODEL_PATH]:
    if not required_path.exists():
        raise FileNotFoundError(f"Required checkpoint path does not exist: {required_path}")

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print(f"Output root: {OUTPUT_ROOT}")
print(f"Model dtype: {MODEL_DTYPE}")


In [ ]:
@dataclass(frozen=True)
class NaiveMergeConfig:
    """Configuration for baseline linear task-vector merge.

    Args:
        if_weight: Weight assigned to IF task vector delta.
        math_weight: Weight assigned to Math task vector delta.
    """

    if_weight: float
    math_weight: float


@dataclass(frozen=True)
class CoordinateGateConfig:
    """Configuration for coordinate-wise gating merge.

    Args:
        a: Magnitude gain for IF delta (`|Δ_if|`).
        b: Magnitude gain for Math delta (`|Δ_math|`).
        c: Gain for sign agreement feature.
        min_gate: Lower clipping bound for numerical stability.
        max_gate: Upper clipping bound for numerical stability.
    """

    a: float
    b: float
    c: float
    min_gate: float = 0.01
    max_gate: float = 0.99


@dataclass(frozen=True)
class NormAwareConfig:
    """Configuration for norm-aware nonlinear blending.

    Args:
        mode: Saturation mode, either `tanh` or `clip`.
        saturation_value: Threshold/temperature for saturation.
        norm_power: Exponent for inverse-norm weighting.
        eps: Small constant to avoid divide-by-zero.
    """

    mode: str
    saturation_value: float
    norm_power: float = 1.0
    eps: float = 1e-8


@dataclass(frozen=True)
class SubspaceMergeConfig:
    """Configuration for low-dimensional subspace-aware merge.

    Args:
        rank: Number of basis axes to use (`1`=common only, `2`=common+orthogonal).
        common_strength: Scale factor for common component.
        shared_orth_strength: Scale factor when orthogonal coefficients agree in sign.
        conflict_strength: Scale factor when orthogonal coefficients disagree in sign.
        if_conflict_weight: IF weight in conflict branch.
        math_conflict_weight: Math weight in conflict branch.
        eps: Small constant for stable normalization.
    """

    rank: int
    common_strength: float
    shared_orth_strength: float
    conflict_strength: float
    if_conflict_weight: float
    math_conflict_weight: float
    eps: float = 1e-8


@dataclass(frozen=True)
class ExperimentSpec:
    """Container that maps an experiment name to one merge family and config.

    Args:
        name: Output directory suffix and human-readable experiment identifier.
        family: Merge method family (`naive`, `coordinate_gate`, `norm_aware`, `subspace`).
        config: One of merge config dataclasses defined above.
    """

    name: str
    family: str
    config: object


In [ ]:
def now_iso() -> str:
    """Return UTC timestamp for metadata versioning.

    Returns:
        ISO-8601 UTC timestamp string.
    """

    return datetime.utcnow().isoformat(timespec="seconds") + "Z"


def save_json(payload: Mapping, output_path: Path) -> None:
    """Persist a JSON-serializable payload with pretty formatting.

    Args:
        payload: Mapping that can be serialized by `json.dump`.
        output_path: Destination JSON file path.

    Returns:
        None. Writes the file to disk.
    """

    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("w", encoding="utf-8") as file:
        json.dump(payload, file, indent=2, ensure_ascii=False)


def load_causal_lm(
    model_name_or_path: str | Path,
    torch_dtype: torch.dtype,
) -> Tuple[AutoModelForCausalLM, AutoTokenizer]:
    """Load a Causal LM + tokenizer on CPU with stable options.

    Args:
        model_name_or_path: HF model ID or local checkpoint path.
        torch_dtype: Dtype used during model loading.

    Returns:
        Tuple `(model, tokenizer)`.
    """

    resolved = str(model_name_or_path)
    model = AutoModelForCausalLM.from_pretrained(
        resolved,
        torch_dtype=torch_dtype,
        device_map="cpu",
        low_cpu_mem_usage=True,
        trust_remote_code=True,
    )
    tokenizer = AutoTokenizer.from_pretrained(
        resolved,
        trust_remote_code=True,
    )
    return model, tokenizer


def validate_state_dict_compatibility(
    base_state: Mapping[str, torch.Tensor],
    tuned_state: Mapping[str, torch.Tensor],
    base_name: str,
    tuned_name: str,
) -> None:
    """Validate exact key/shape compatibility between two state dicts.

    Args:
        base_state: Reference state dictionary.
        tuned_state: Candidate state dictionary to compare.
        base_name: Human-readable label for reference model.
        tuned_name: Human-readable label for candidate model.

    Returns:
        None. Raises `ValueError` on mismatch.
    """

    base_keys = set(base_state.keys())
    tuned_keys = set(tuned_state.keys())

    if base_keys != tuned_keys:
        missing_in_tuned = sorted(base_keys - tuned_keys)
        missing_in_base = sorted(tuned_keys - base_keys)
        raise ValueError(
            f"State dict keys mismatch: {base_name} vs {tuned_name}. "
            f"Missing in tuned={missing_in_tuned[:5]}, missing in base={missing_in_base[:5]}"
        )

    for key in base_state.keys():
        if base_state[key].shape != tuned_state[key].shape:
            raise ValueError(
                "State dict shape mismatch at key "
                f"'{key}': {base_name}={tuple(base_state[key].shape)}, "
                f"{tuned_name}={tuple(tuned_state[key].shape)}"
            )


def cleanup_memory() -> None:
    """Run explicit host/device cache cleanup for long notebook sessions.

    Returns:
        None. Frees Python and CUDA caches when available.
    """

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


In [ ]:
def compute_task_vector(
    base_tensor: torch.Tensor,
    tuned_tensor: torch.Tensor,
) -> torch.Tensor:
    """Compute task vector delta (`tuned - base`) in float32 precision.

    Args:
        base_tensor: Base model tensor.
        tuned_tensor: Task-tuned model tensor.

    Returns:
        Float32 tensor delta with the same shape as inputs.
    """

    base_fp32 = base_tensor.detach().to(torch.float32)
    tuned_fp32 = tuned_tensor.detach().to(torch.float32)
    return tuned_fp32 - base_fp32


def summarize_task_vectors(
    base_model: AutoModelForCausalLM,
    if_model: AutoModelForCausalLM,
    math_model: AutoModelForCausalLM,
) -> Dict[str, float]:
    """Compute global norm/alignment summary for IF and Math task vectors.

    Args:
        base_model: Base checkpoint model.
        if_model: IF tuned checkpoint model.
        math_model: Math tuned checkpoint model.

    Returns:
        Dictionary containing global L2 norms, cosine similarity, and parameter count.
    """

    base_state = base_model.state_dict()
    if_state = if_model.state_dict()
    math_state = math_model.state_dict()

    validate_state_dict_compatibility(base_state, if_state, "base", "if_model")
    validate_state_dict_compatibility(base_state, math_state, "base", "math_model")

    if_sq_sum = 0.0
    math_sq_sum = 0.0
    dot_sum = 0.0
    floating_param_count = 0

    with torch.no_grad():
        for name in tqdm(base_state.keys(), desc="Task-vector summary"):
            base_tensor = base_state[name]
            if not torch.is_floating_point(base_tensor):
                continue

            if_delta = compute_task_vector(base_tensor, if_state[name])
            math_delta = compute_task_vector(base_tensor, math_state[name])

            # Accumulate global squared norms and cross-dot for cosine alignment.
            if_sq_sum += float(torch.sum(if_delta * if_delta).item())
            math_sq_sum += float(torch.sum(math_delta * math_delta).item())
            dot_sum += float(torch.sum(if_delta * math_delta).item())
            floating_param_count += int(base_tensor.numel())

    if_norm = if_sq_sum ** 0.5
    math_norm = math_sq_sum ** 0.5
    cosine = dot_sum / (if_norm * math_norm + 1e-12)

    return {
        "if_delta_l2": if_norm,
        "math_delta_l2": math_norm,
        "if_math_dot": dot_sum,
        "if_math_cosine": cosine,
        "floating_param_count": floating_param_count,
    }


In [ ]:
def naive_merge_delta(
    if_delta: torch.Tensor,
    math_delta: torch.Tensor,
    config: NaiveMergeConfig,
) -> Tuple[torch.Tensor, Dict[str, float]]:
    """Compute baseline linear merge delta for one tensor.

    Args:
        if_delta: IF task vector delta tensor.
        math_delta: Math task vector delta tensor.
        config: Baseline merge weights.

    Returns:
        Tuple of `(merged_delta, stats_dict)`.
    """

    merged_delta = (config.if_weight * if_delta) + (config.math_weight * math_delta)
    return merged_delta, {"if_weight": float(config.if_weight), "math_weight": float(config.math_weight)}


def compute_sign_agreement(if_delta: torch.Tensor, math_delta: torch.Tensor) -> torch.Tensor:
    """Compute coordinate-level sign agreement signal in `{-1, 0, +1}`.

    Args:
        if_delta: IF task vector delta tensor.
        math_delta: Math task vector delta tensor.

    Returns:
        Tensor with +1 for same sign, -1 for opposite sign, 0 when either side is zero.
    """

    product = if_delta * math_delta
    agreement = torch.zeros_like(product)
    agreement = torch.where(product > 0, torch.ones_like(product), agreement)
    agreement = torch.where(product < 0, -torch.ones_like(product), agreement)
    return agreement


def coordinate_gated_merge_delta(
    if_delta: torch.Tensor,
    math_delta: torch.Tensor,
    config: CoordinateGateConfig,
) -> Tuple[torch.Tensor, Dict[str, float]]:
    """Compute coordinate-wise gated nonlinear merge delta for one tensor.

    Args:
        if_delta: IF task vector delta tensor.
        math_delta: Math task vector delta tensor.
        config: Coordinate gating hyperparameters.

    Returns:
        Tuple of `(merged_delta, stats_dict)` where stats include gate averages.
    """

    agreement = compute_sign_agreement(if_delta, math_delta)

    # Follow the requested formula exactly while keeping gate bounded away
    # from 0/1 to prevent fully hard routing in early experiments.
    logits = (config.a * torch.abs(if_delta)) - (config.b * torch.abs(math_delta)) + (config.c * agreement)
    gate = torch.sigmoid(logits)
    gate = torch.clamp(gate, min=config.min_gate, max=config.max_gate)

    merged_delta = (gate * if_delta) + ((1.0 - gate) * math_delta)

    stats = {
        "gate_mean": float(gate.mean().item()),
        "gate_min": float(gate.min().item()),
        "gate_max": float(gate.max().item()),
        "disagree_rate": float((agreement < 0).float().mean().item()),
    }
    return merged_delta, stats


def saturate_delta(
    delta: torch.Tensor,
    mode: str,
    saturation_value: float,
) -> torch.Tensor:
    """Saturate large deltas to reduce dominance of high-magnitude updates.

    Args:
        delta: Input task-vector delta tensor.
        mode: Either `tanh` or `clip`.
        saturation_value: Positive threshold/temperature value.

    Returns:
        Saturated delta tensor with the same shape as input.
    """

    if saturation_value <= 0:
        raise ValueError(f"saturation_value must be positive, got {saturation_value}")

    if mode == "tanh":
        return saturation_value * torch.tanh(delta / saturation_value)
    if mode == "clip":
        return torch.clamp(delta, min=-saturation_value, max=saturation_value)

    raise ValueError(f"Unsupported saturation mode: {mode}")


def norm_aware_merge_delta(
    if_delta: torch.Tensor,
    math_delta: torch.Tensor,
    config: NormAwareConfig,
) -> Tuple[torch.Tensor, Dict[str, float]]:
    """Compute nonlinear merge using saturation + inverse-norm weighting.

    Args:
        if_delta: IF task vector delta tensor.
        math_delta: Math task vector delta tensor.
        config: Norm-aware merge hyperparameters.

    Returns:
        Tuple of `(merged_delta, stats_dict)` including per-tensor scalar weights.
    """

    if_sat = saturate_delta(if_delta, mode=config.mode, saturation_value=config.saturation_value)
    math_sat = saturate_delta(math_delta, mode=config.mode, saturation_value=config.saturation_value)

    # Inverse-norm weighting downweights whichever task produces a larger
    # saturated update magnitude on this tensor.
    if_norm = float(torch.linalg.vector_norm(if_sat).item())
    math_norm = float(torch.linalg.vector_norm(math_sat).item())

    if_inv = 1.0 / ((if_norm + config.eps) ** config.norm_power)
    math_inv = 1.0 / ((math_norm + config.eps) ** config.norm_power)

    inv_sum = if_inv + math_inv
    if_weight = if_inv / inv_sum
    math_weight = math_inv / inv_sum

    merged_delta = (if_weight * if_sat) + (math_weight * math_sat)

    stats = {
        "if_weight": float(if_weight),
        "math_weight": float(math_weight),
        "if_norm_sat": if_norm,
        "math_norm_sat": math_norm,
    }
    return merged_delta, stats


def build_two_axis_subspace_basis(
    if_delta_flat: torch.Tensor,
    math_delta_flat: torch.Tensor,
    eps: float,
) -> Tuple[Optional[torch.Tensor], Optional[torch.Tensor]]:
    """Construct common and orthogonal axes from two task vectors.

    Args:
        if_delta_flat: Flattened IF task vector.
        math_delta_flat: Flattened Math task vector.
        eps: Numerical stability constant for norm checks.

    Returns:
        Tuple `(common_axis, orth_axis)`. `orth_axis` can be `None` if rank-1.
    """

    # Common axis starts from the sum direction; it emphasizes agreement.
    common_seed = if_delta_flat + math_delta_flat
    common_norm = torch.linalg.vector_norm(common_seed)

    if float(common_norm.item()) <= eps:
        common_seed = if_delta_flat
        common_norm = torch.linalg.vector_norm(common_seed)

    if float(common_norm.item()) <= eps:
        # Degenerate tensor where both deltas are effectively zero.
        return None, None

    common_axis = common_seed / (common_norm + eps)

    # Orthogonal axis captures disagreement/conflict not explained by common axis.
    residual = if_delta_flat - torch.dot(if_delta_flat, common_axis) * common_axis
    residual_norm = torch.linalg.vector_norm(residual)

    if float(residual_norm.item()) <= eps:
        residual = math_delta_flat - torch.dot(math_delta_flat, common_axis) * common_axis
        residual_norm = torch.linalg.vector_norm(residual)

    if float(residual_norm.item()) <= eps:
        return common_axis, None

    orth_axis = residual / (residual_norm + eps)
    return common_axis, orth_axis


def subspace_merge_delta(
    if_delta: torch.Tensor,
    math_delta: torch.Tensor,
    config: SubspaceMergeConfig,
) -> Tuple[torch.Tensor, Dict[str, float]]:
    """Merge two deltas using rank-1/2 subspace decomposition.

    Args:
        if_delta: IF task vector delta tensor.
        math_delta: Math task vector delta tensor.
        config: Subspace merge configuration.

    Returns:
        Tuple of `(merged_delta, stats_dict)` with basis coefficient diagnostics.
    """

    if config.rank not in (1, 2):
        raise ValueError(f"Subspace rank must be 1 or 2, got {config.rank}")

    if_flat = if_delta.reshape(-1)
    math_flat = math_delta.reshape(-1)

    common_axis, orth_axis = build_two_axis_subspace_basis(if_flat, math_flat, eps=config.eps)
    if common_axis is None:
        return torch.zeros_like(if_delta), {"used_orth_axis": 0.0, "common_coeff": 0.0, "orth_coeff": 0.0}

    # Project both task vectors into common axis and merge with dedicated strength.
    if_common_coeff = torch.dot(if_flat, common_axis)
    math_common_coeff = torch.dot(math_flat, common_axis)
    merged_common_coeff = config.common_strength * 0.5 * (if_common_coeff + math_common_coeff)

    merged_flat = merged_common_coeff * common_axis
    merged_orth_coeff = torch.tensor(0.0, device=if_delta.device, dtype=if_delta.dtype)
    used_orth_axis = 0.0

    if config.rank == 2 and orth_axis is not None:
        used_orth_axis = 1.0
        if_orth_coeff = torch.dot(if_flat, orth_axis)
        math_orth_coeff = torch.dot(math_flat, orth_axis)

        # If orthogonal coefficients agree in sign, treat them as additional
        # shared signal; otherwise shrink conflict branch separately.
        if float((if_orth_coeff * math_orth_coeff).item()) >= 0.0:
            merged_orth_coeff = config.shared_orth_strength * 0.5 * (if_orth_coeff + math_orth_coeff)
        else:
            merged_orth_coeff = config.conflict_strength * (
                config.if_conflict_weight * if_orth_coeff
                + config.math_conflict_weight * math_orth_coeff
            )

        merged_flat = merged_flat + (merged_orth_coeff * orth_axis)

    stats = {
        "used_orth_axis": used_orth_axis,
        "common_coeff": float(merged_common_coeff.item()),
        "orth_coeff": float(merged_orth_coeff.item()),
    }
    return merged_flat.view_as(if_delta), stats


In [ ]:
def merge_base_with_strategy_inplace(
    base_model: AutoModelForCausalLM,
    if_model: AutoModelForCausalLM,
    math_model: AutoModelForCausalLM,
    spec: ExperimentSpec,
) -> Dict[str, float]:
    """Apply one merge strategy to `base_model` in-place and collect diagnostics.

    Args:
        base_model: Base model that will receive merged parameters.
        if_model: IF tuned model.
        math_model: Math tuned model.
        spec: Experiment specification containing family and config.

    Returns:
        Dictionary with aggregated scalar diagnostics over all tensors.
    """

    base_state = base_model.state_dict()
    if_state = if_model.state_dict()
    math_state = math_model.state_dict()

    validate_state_dict_compatibility(base_state, if_state, "base", "if_model")
    validate_state_dict_compatibility(base_state, math_state, "base", "math_model")

    aggregate_stats: Dict[str, float] = {
        "tensor_count": 0.0,
        "gate_mean_acc": 0.0,
        "if_weight_acc": 0.0,
        "math_weight_acc": 0.0,
        "used_orth_axis_acc": 0.0,
    }

    with torch.no_grad():
        for name in tqdm(base_state.keys(), desc=f"Merging ({spec.name})"):
            base_tensor = base_state[name]

            # Keep non-floating tensors unchanged (e.g., integer buffers).
            if not torch.is_floating_point(base_tensor):
                continue

            if_delta = compute_task_vector(base_tensor, if_state[name])
            math_delta = compute_task_vector(base_tensor, math_state[name])

            if spec.family == "naive":
                merged_delta, tensor_stats = naive_merge_delta(if_delta, math_delta, spec.config)
            elif spec.family == "coordinate_gate":
                merged_delta, tensor_stats = coordinate_gated_merge_delta(if_delta, math_delta, spec.config)
            elif spec.family == "norm_aware":
                merged_delta, tensor_stats = norm_aware_merge_delta(if_delta, math_delta, spec.config)
            elif spec.family == "subspace":
                merged_delta, tensor_stats = subspace_merge_delta(if_delta, math_delta, spec.config)
            else:
                raise ValueError(f"Unsupported merge family: {spec.family}")

            merged_parameter = base_tensor.detach().to(torch.float32) + merged_delta
            base_tensor.copy_(merged_parameter.to(dtype=base_tensor.dtype))

            # Aggregate only key scalar diagnostics that are comparable across tensors.
            aggregate_stats["tensor_count"] += 1.0
            aggregate_stats["gate_mean_acc"] += float(tensor_stats.get("gate_mean", 0.0))
            aggregate_stats["if_weight_acc"] += float(tensor_stats.get("if_weight", 0.0))
            aggregate_stats["math_weight_acc"] += float(tensor_stats.get("math_weight", 0.0))
            aggregate_stats["used_orth_axis_acc"] += float(tensor_stats.get("used_orth_axis", 0.0))

    tensor_count = max(aggregate_stats["tensor_count"], 1.0)
    aggregate_stats["gate_mean_avg"] = aggregate_stats["gate_mean_acc"] / tensor_count
    aggregate_stats["if_weight_avg"] = aggregate_stats["if_weight_acc"] / tensor_count
    aggregate_stats["math_weight_avg"] = aggregate_stats["math_weight_acc"] / tensor_count
    aggregate_stats["used_orth_axis_rate"] = aggregate_stats["used_orth_axis_acc"] / tensor_count

    return aggregate_stats


def save_merged_artifacts(
    merged_model: AutoModelForCausalLM,
    tokenizer: AutoTokenizer,
    output_dir: Path,
    metadata: Mapping,
) -> None:
    """Save merged model, tokenizer, and metadata to disk.

    Args:
        merged_model: Merged checkpoint model.
        tokenizer: Tokenizer aligned with checkpoint.
        output_dir: Destination folder.
        metadata: JSON metadata payload for reproducibility.

    Returns:
        None. Writes files to disk.
    """

    output_dir.mkdir(parents=True, exist_ok=True)
    merged_model.save_pretrained(output_dir, safe_serialization=True)
    tokenizer.save_pretrained(output_dir)
    save_json(metadata, output_dir / "merge_metadata.json")
    print(f"Saved merged checkpoint: {output_dir}")


In [ ]:
# --------------------------------------------------------------------------------------
# Experiment registry
# --------------------------------------------------------------------------------------
# Baseline from `merging.ipynb` for direct comparison.
baseline_uniform = ExperimentSpec(
    name="naive_uniform",
    family="naive",
    config=NaiveMergeConfig(if_weight=0.5, math_weight=0.5),
)

coordinate_gate_balanced = ExperimentSpec(
    name="coordinate_gate_balanced",
    family="coordinate_gate",
    config=CoordinateGateConfig(a=96.0, b=96.0, c=1.5, min_gate=0.01, max_gate=0.99),
)

coordinate_gate_if_bias = ExperimentSpec(
    name="coordinate_gate_if_bias",
    family="coordinate_gate",
    config=CoordinateGateConfig(a=128.0, b=80.0, c=1.0, min_gate=0.01, max_gate=0.99),
)

norm_aware_tanh = ExperimentSpec(
    name="norm_aware_tanh",
    family="norm_aware",
    config=NormAwareConfig(mode="tanh", saturation_value=0.02, norm_power=1.0, eps=1e-8),
)

norm_aware_clip = ExperimentSpec(
    name="norm_aware_clip",
    family="norm_aware",
    config=NormAwareConfig(mode="clip", saturation_value=0.02, norm_power=1.0, eps=1e-8),
)

subspace_rank1 = ExperimentSpec(
    name="subspace_rank1_common_only",
    family="subspace",
    config=SubspaceMergeConfig(
        rank=1,
        common_strength=1.0,
        shared_orth_strength=0.0,
        conflict_strength=0.0,
        if_conflict_weight=0.5,
        math_conflict_weight=0.5,
        eps=1e-8,
    ),
)

subspace_rank2_conflict_shrink = ExperimentSpec(
    name="subspace_rank2_conflict_shrink",
    family="subspace",
    config=SubspaceMergeConfig(
        rank=2,
        common_strength=1.0,
        shared_orth_strength=0.8,
        conflict_strength=0.3,
        if_conflict_weight=0.5,
        math_conflict_weight=0.5,
        eps=1e-8,
    ),
)

EXPERIMENTS: List[ExperimentSpec] = [
    baseline_uniform,
    coordinate_gate_balanced,
    coordinate_gate_if_bias,
    norm_aware_tanh,
    norm_aware_clip,
    subspace_rank1,
    subspace_rank2_conflict_shrink,
]

print("Registered experiments:")
for spec in EXPERIMENTS:
    print(f" - {spec.name} ({spec.family})")


In [ ]:
# --------------------------------------------------------------------------------------
# Load tuned models once and compute task-vector summary
# --------------------------------------------------------------------------------------
if_model, _ = load_causal_lm(IF_MODEL_PATH, MODEL_DTYPE)
math_model, _ = load_causal_lm(MATH_MODEL_PATH, MODEL_DTYPE)

analysis_base_model, _ = load_causal_lm(BASE_MODEL_ID, MODEL_DTYPE)
task_vector_summary = summarize_task_vectors(analysis_base_model, if_model, math_model)

task_vector_metadata = {
    "created_at_utc": now_iso(),
    "base_model": BASE_MODEL_ID,
    "if_model": str(IF_MODEL_PATH),
    "math_model": str(MATH_MODEL_PATH),
    "summary": task_vector_summary,
}
save_json(task_vector_metadata, OUTPUT_ROOT / "task_vector_summary.json")

print("Task-vector summary:")
for key, value in task_vector_summary.items():
    print(f" - {key}: {value}")

del analysis_base_model
cleanup_memory()


In [ ]:
# --------------------------------------------------------------------------------------
# Run all merge experiments
# --------------------------------------------------------------------------------------
run_metadata: List[Dict[str, object]] = []

for spec in EXPERIMENTS:
    print("\n" + "=" * 100)
    print(f"Running experiment: {spec.name} ({spec.family})")

    base_model, base_tokenizer = load_causal_lm(BASE_MODEL_ID, MODEL_DTYPE)

    aggregate_stats = merge_base_with_strategy_inplace(
        base_model=base_model,
        if_model=if_model,
        math_model=math_model,
        spec=spec,
    )

    output_dir = OUTPUT_ROOT / spec.name
    metadata = {
        "created_at_utc": now_iso(),
        "experiment_name": spec.name,
        "family": spec.family,
        "config": asdict(spec.config),
        "base_model": BASE_MODEL_ID,
        "if_model": str(IF_MODEL_PATH),
        "math_model": str(MATH_MODEL_PATH),
        "aggregate_stats": aggregate_stats,
        "task_vector_summary_path": str(OUTPUT_ROOT / "task_vector_summary.json"),
        "output_dir": str(output_dir),
    }

    save_merged_artifacts(
        merged_model=base_model,
        tokenizer=base_tokenizer,
        output_dir=output_dir,
        metadata=metadata,
    )

    run_metadata.append(metadata)

    # Explicit cleanup between experiments to keep memory usage stable.
    del base_model
    del base_tokenizer
    cleanup_memory()

save_json(
    {
        "created_at_utc": now_iso(),
        "output_root": str(OUTPUT_ROOT),
        "experiment_count": len(run_metadata),
        "experiments": run_metadata,
    },
    OUTPUT_ROOT / "experiment_index.json",
)

print("\nCompleted all experiments.")
print(f"Index file: {OUTPUT_ROOT / 'experiment_index.json'}")


In [ ]:
# --------------------------------------------------------------------------------------
# Final cleanup
# --------------------------------------------------------------------------------------
del if_model
del math_model
cleanup_memory()

print("Released IF/Math model objects and completed notebook run.")
